In [ ]:
# train with https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis

In [13]:
import kagglehub, pandas as pd, re
from pathlib import Path
from datasets import Dataset, ClassLabel

ID2LABEL = {0:"negative", 1:"neutral", 2:"positive"}
LABEL2ID = {v:k for k,v in ID2LABEL.items()}

def _parse_phrasebank_txt(p: Path) -> pd.DataFrame:
    """
    Lines look like:
      positive\tSome sentence...
      negative\tAnother sentence...
      neutral \t...
    """
    rows = []
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line: 
                continue
            # split on first tab or first space-colon patterns seen in some mirrors
            m = re.match(r"^(positive|negative|neutral)\s*[,\t;:]\s*(.+)$", line, flags=re.I)
            if not m:
                parts = re.split(r"\t", line, maxsplit=1)
                if len(parts) == 2 and parts[0].lower() in {"positive","negative","neutral"}:
                    m = (parts[0], parts[1])
                else:
                    continue
            lab = (m.group(1) if hasattr(m, "group") else m[0]).strip().lower()
            txt = (m.group(2) if hasattr(m, "group") else m[1]).strip()
            rows.append({"text": txt, "label": {"negative":0,"neutral":1,"positive":2}[lab]})
    return pd.DataFrame(rows)

def _try_csv(p: Path) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(p, encoding="utf-8")
    except Exception:
        try:
            df = pd.read_csv(p, encoding="ISO-8859-1")
        except Exception:
            return None
    cols = {c.lower(): c for c in df.columns}
    # common columns: sentence/text, sentiment/label
    text_col  = cols.get("sentence") or cols.get("text") or cols.get("content")
    label_col = cols.get("sentiment") or cols.get("label") or cols.get("class")
    if not text_col or not label_col:
        return None
    s = df[[text_col, label_col]].rename(columns={text_col:"text", label_col:"label"}).dropna()
    # normalize labels
    s["label"] = (
        pd.to_numeric(s["label"], errors="coerce")
          .map({-1:0, 0:1, 1:2})  # common {-1,0,1}
          .fillna(s["label"].astype(str).str.lower().map({"negative":0,"neutral":1,"positive":2}))
          .astype("Int64")
    )
    s = s.dropna(subset=["label"]).astype({"label":"int"})
    s["text"] = s["text"].astype(str).str.replace(r"\s+"," ", regex=True).str.strip()
    return s[["text","label"]]

def load_phrasebank_finance() -> Dataset:
    path = Path(kagglehub.dataset_download("sbhatti/financial-sentiment-analysis"))
    # Typical layout has a FinancialPhraseBank folder
    candidates = list(path.rglob("Sentences_*Agree.txt")) + list(path.rglob("*.csv"))
    if not candidates:
        raise RuntimeError("Could not find PhraseBank files in the Kaggle folder.")
    frames = []
    for p in candidates:
        if p.suffix.lower() == ".txt":
            try:
                frames.append(_parse_phrasebank_txt(p))
            except Exception:
                pass
        elif p.suffix.lower() == ".csv":
            df = _try_csv(p)
            if df is not None:
                frames.append(df)
    if not frames:
        raise RuntimeError("Parsing failed for all files.")
    df = pd.concat(frames, ignore_index=True)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"[PhraseBank] rows={len(df):,}  label_counts={df['label'].value_counts().to_dict()}")
    ds = Dataset.from_pandas(df[["text","label"]], preserve_index=False)
    ds = ds.cast_column("label", ClassLabel(names=[ID2LABEL[i] for i in range(3)]))
    return ds


In [7]:
from datasets import load_dataset

# Existing datasets you already used (examples)
# fin_ds  = load_financial_news()            # your previous loader
# sst3    = load_sst2_as_three_class()       # your previous helper (optional)
# cosmos  = load_cosmos98_twitter_reddit()   # if you added this previously

kashish = load_kashish_social_media()

# Combine (add kashish on top of what you already had)
# example: combined = concatenate_datasets([fin_ds, cosmos, kashish])
combined = kashish  # <- if you just want to try only this new dataset now

# Stratified split
combined = combined.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, val_ds = combined["train"], combined["test"]
print(train_ds, val_ds)


100%|██████████████████████████████████████| 50.9k/50.9k [00:00<00:00, 28.1MB/s]

Extracting files...
[kashish] rows=140  label_counts={2: 103, 0: 19, 1: 18}


Casting the dataset:   0%|          | 0/140 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 126
}) Dataset({
    features: ['text', 'label'],
    num_rows: 14
})


In [15]:
from datasets import concatenate_datasets
from transformers import AutoTokenizer

phrasebank = load_phrasebank_finance()

# If you already have another training dataset, concat:
# combined = concatenate_datasets([your_existing_ds, phrasebank])
combined = phrasebank

# Stratified split
combined = combined.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, val_ds = combined["train"], combined["test"]

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
def tok_fn(batch):
    return tok(batch["text"], padding="max_length", truncation=True, max_length=128)

tok_train = train_ds.map(tok_fn, batched=True)
tok_eval  = val_ds.map(tok_fn, batched=True)


100%|████████████████████████████████████████| 276k/276k [00:00<00:00, 5.07MB/s]

Extracting files...
[PhraseBank] rows=5,322  label_counts={1: 2878, 2: 1852, 0: 592}


Casting the dataset:   0%|          | 0/5322 [00:00<?, ? examples/s]

/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/4789 [00:00<?, ? examples/s]

Map:   0%|          | 0/533 [00:00<?, ? examples/s]

In [17]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    "./deberta-financial",   # resume from your current directory
    num_labels=3
)

acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")
def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {
        "accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=p.label_ids, average="macro")["f1"]
    }

args = TrainingArguments(
    output_dir="out-continued-phrasebank",
    learning_rate=1e-5,                # softer LR—PhraseBank is small but high quality
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",             # v5
    eval_steps=200,
    save_strategy="steps",             # v5
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_eval,
    processing_class=tok,              # v5: replaces tokenizer=
    compute_metrics=metrics,
)

trainer.train()
model.save_pretrained("./deberta-financial")
tok.save_pretrained("./deberta-financial")


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
200,0.328800,0.307635,0.866792,0.819020
400,0.220000,0.314854,0.868668,0.826990
600,0.220900,0.311518,0.893058,0.845210


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


('./deberta-financial/tokenizer_config.json',
 './deberta-financial/special_tokens_map.json',
 './deberta-financial/spm.model',
 './deberta-financial/added_tokens.json',
 './deberta-financial/tokenizer.json')

In [19]:
# https://www.kaggle.com/datasets/waseemalastal/sentiment-for-financial-news-dataset
# https://www.kaggle.com/datasets/antobenedetti/finance-news-sentiments
# https://www.kaggle.com/datasets/aravsood7/sentiment-analysis-labelled-financial-news-data
# https://www.kaggle.com/datasets/sixlack/finaldf

In [ ]:
# https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis
# https://www.kaggle.com/datasets/mdismielhossenabir/sentiment-analysis

In [ ]:
# https://www.kaggle.com/datasets/aravsood7/sentiment-analysis-labelled-financial-news-data

In [19]:
import kagglehub, pandas as pd
from pathlib import Path

root = Path(kagglehub.dataset_download("aravsood7/sentiment-analysis-labelled-financial-news-data"))
print("DATASET ROOT:", root)

for p in sorted(root.rglob("*"))[:40]:
    if p.is_file():
        print(p.relative_to(root))

def peek(p, n=3):
    for enc in ("utf-8","ISO-8859-1"):
        try:
            df = pd.read_csv(p, encoding=enc)
            print(f"\n--- {p.name} [{enc}] ---")
            print("columns:", list(df.columns))
            print(df.head(n))
            return
        except Exception:
            pass

for p in sorted(root.rglob("*.csv"))[:5]:
    peek(p)


100%|████████████████████████████████████████| 310k/310k [00:00<00:00, 3.88MB/s]

Extracting files...
DATASET ROOT: /Users/rickliu/.cache/kagglehub/datasets/aravsood7/sentiment-analysis-labelled-financial-news-data/versions/1
Fin_Cleaned.csv

--- Fin_Cleaned.csv [utf-8] ---
columns: ['Date_published', 'Headline', 'Synopsis', 'Full_text', 'Final Status']
  Date_published                                           Headline  \
0     2022-06-21  Banks holding on to subsidy share, say payment...   
1     2022-04-19  Digitally ready Bank of Baroda aims to click o...   
2     2022-05-27  Karnataka attracted investment commitment of R...   

                                            Synopsis  \
0  The companies have written to the National Pay...   
1  At present, 50% of the bank's retail loans are...   
2  Karnataka is at the forefront in attracting in...   

                                           Full_text Final Status  
0  ReutersPayments companies and banks are at log...     Negative  
1  AgenciesThe bank presently has 20 million acti...     Positive  
2  PTIKarnat

In [21]:
# --- loader for aravsood7 dataset (3-class) ---
import kagglehub, pandas as pd, re
from pathlib import Path
from datasets import Dataset, ClassLabel

ID2LABEL = {0:"negative", 1:"neutral", 2:"positive"}
MAP = {"negative":0, "neutral":1, "other":1, "positive":2}  # treat "Other" as neutral

def load_aravsood_fin_news():
    root = Path(kagglehub.dataset_download("aravsood7/sentiment-analysis-labelled-financial-news-data"))
    # main file is typically a single CSV with columns like:
    # Date_published, Headline, Synopsis, Full_text, Final Status
    # Prefer Full_text; fall back to Headline+Synopsis if missing
    df = None
    for enc in ("utf-8","ISO-8859-1"):
        try:
            df = pd.read_csv(next(root.rglob("*.csv")), encoding=enc)
            break
        except Exception:
            pass
    assert df is not None and not df.empty

    text = (df["Full_text"].fillna("") + " " + df["Headline"].fillna("") + " " + df["Synopsis"].fillna("")).str.strip()
    lab  = df["Final Status"].astype(str).str.lower().str.strip().map(MAP)

    use = pd.DataFrame({"text": text, "label": lab}).dropna()
    use["text"] = use["text"].astype(str).str.replace(r"\s+"," ", regex=True)
    use = use.drop_duplicates(subset=["text"])

    ds = Dataset.from_pandas(use, preserve_index=False)
    ds = ds.cast_column("label", ClassLabel(names=[ID2LABEL[i] for i in range(3)]))
    return ds


In [23]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import DatasetDict
import numpy as np, evaluate

ds = load_aravsood_fin_news()
ds = ds.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tok_fn(b): return tok(b["text"], padding="max_length", truncation=True, max_length=128)
tok_train, tok_eval = ds["train"].map(tok_fn, batched=True), ds["test"].map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("./deberta-financial", num_labels=3)

acc, f1 = evaluate.load("accuracy"), evaluate.load("f1")
def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
            "f1_macro": f1.compute(predictions=preds, references=p.label_ids, average="macro")["f1"]}

args = TrainingArguments(
    output_dir="out-continued-aravsood",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=tok_train, eval_dataset=tok_eval,
                  processing_class=tok, compute_metrics=metrics)

trainer.train()
model.save_pretrained("./deberta-financial")
tok.save_pretrained("./deberta-financial")


Casting the dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


('./deberta-financial/tokenizer_config.json',
 './deberta-financial/special_tokens_map.json',
 './deberta-financial/spm.model',
 './deberta-financial/added_tokens.json',
 './deberta-financial/tokenizer.json')

In [ ]:
# https://www.kaggle.com/datasets/sixlack/finaldf

In [25]:
import kagglehub, pandas as pd, re
from pathlib import Path
from datasets import Dataset, ClassLabel

ID2LABEL = {0:"negative", 1:"neutral", 2:"positive"}

def load_sixlack_finaldf(map_other_to_neutral: bool = True) -> Dataset:
    root = Path(kagglehub.dataset_download("sixlack/finaldf"))
    # The file is named like 'data-3.csv'
    csv = next(root.rglob("*.csv"))
    # Read robustly
    for enc in ("utf-8", "ISO-8859-1"):
        try:
            df = pd.read_csv(csv, encoding=enc)
            break
        except Exception:
            pass
    assert df is not None and not df.empty, "Could not read CSV."

    # Column names in the preview: 'Sentence', 'Sentiment'
    text = df["Sentence"].astype(str)
    lab  = df["Sentiment"].astype(str).str.lower().str.strip()

    if map_other_to_neutral:
        lab = lab.replace({"other": "neutral"})
        label_map = {"negative":0, "neutral":1, "positive":2}
    else:
        # drop 'other'
        keep = lab.isin(["negative","neutral","positive"])
        text, lab = text[keep], lab[keep]
        label_map = {"negative":0, "neutral":1, "positive":2}

    y = lab.map(label_map)
    use = pd.DataFrame({"text": text, "label": y}).dropna()
    use["text"] = use["text"].str.replace(r"\s+"," ", regex=True).str.strip()
    use = use.drop_duplicates(subset=["text"])

    ds = Dataset.from_pandas(use, preserve_index=False)
    ds = ds.cast_column("label", ClassLabel(names=[ID2LABEL[i] for i in range(3)]))
    print(f"[sixlack] rows={len(ds):,}  counts={dict(pd.Series(ds['label']).value_counts())}")
    return ds


In [27]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

ds = load_sixlack_finaldf(map_other_to_neutral=True)
ds = ds.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, val_ds = ds["train"], ds["test"]

tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tok_fn(batch):
    return tok(batch["text"], padding="max_length", truncation=True, max_length=128)

tok_train = train_ds.map(tok_fn, batched=True)
tok_eval  = val_ds.map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "./deberta-financial",   # continue from your current model
    num_labels=3
)

acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")
def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {
        "accuracy": acc.compute(predictions=preds, references=p.label_ids)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=p.label_ids, average="macro")["f1"],
    }

args = TrainingArguments(
    output_dir="out-continued-sixlack",
    learning_rate=1e-5,                 # small LR to avoid forgetting
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",              # Transformers v5
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_eval,
    processing_class=tok,               # v5: replaces tokenizer=
    compute_metrics=metrics,
)

trainer.train()

# overwrite your local model with the improved weights
model.save_pretrained("./deberta-financial")
tok.save_pretrained("./deberta-financial")


100%|████████████████████████████████████████| 276k/276k [00:00<00:00, 11.0MB/s]

Extracting files...


Casting the dataset:   0%|          | 0/5322 [00:00<?, ? examples/s]

[sixlack] rows=5,322  counts={1: 2878, 2: 1852, 0: 592}


/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

Map:   0%|          | 0/533 [00:00<?, ? examples/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
200,0.195200,0.367282,0.874296,0.817280
400,0.113700,0.417940,0.881801,0.834555
600,0.149400,0.425677,0.883677,0.836074


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


('./deberta-financial/tokenizer_config.json',
 './deberta-financial/special_tokens_map.json',
 './deberta-financial/spm.model',
 './deberta-financial/added_tokens.json',
 './deberta-financial/tokenizer.json')